# Document 1 — benign mission through final dataset

Implements the setup document from **section 4 (Fly a Benign Mission)** to the
end. Sections 1-3 of that document are one-time environment installation and are
not repeated here.

| Document section | Cells below |
|---|---|
| 4. Fly a benign mission (Listings 4-5) | manifest, flight, ROS evidence, pose trace |
| 5. Configure the 5G network scenario (Listing 6) | Simu5G run and metric export |
| 6. Build the Python data bridge (Listings 7-8) | `assemble.py` and the validator |
| 7. Add suspicious and malicious scenarios (Table 1) | one run per class |
| 8. Validate and deliver the dataset (Listing 9) | `concat.py` and the validator |
| Final deliverables checklist | closing gate |

This notebook does not reimplement the simulator. It calls the delivered
scripts, so the code that runs here is the code that ships:

- `sim/bridge/export_pose_trace.py` - bag to Simu5G mobility trace
- `sim/bridge/assemble.py` - one row per flow per aggregation window
- `sim/bridge/concat.py` - join runs without changing the schema
- `validate_dataset.py` - the supplied authority on the 44-column contract

**Run order matters.** The flight cell launches Gazebo, PX4 and MAVROS headless
and takes about five minutes. Run cells top to bottom.

In [ ]:
# Setup. Paths and process helpers come from the delivered automation module so
# the launch sequence here is the same one the unattended tests use.
import json, shutil, subprocess, sys, time
from pathlib import Path

import pandas as pd
import yaml

HERE = Path.cwd().resolve()
assert HERE.name == "UAV_5GSim", f"Run this notebook from UAV_5GSim, not {HERE}"
sys.path.insert(0, str(HERE))

import automate_live_duration_tests as auto

SCHEMA = HERE / "schema_reference.json"
VALIDATOR = HERE / "validate_dataset.py"
BRIDGE = HERE / "sim/bridge"
OUT = HERE / "experiments/document1_workflow"
OUT.mkdir(parents=True, exist_ok=True)

DURATION_SEC = 300          # document baseline mission duration
SPEED_MPS = 3.0             # document baseline mission speed
ALTITUDE_M = 30.0
WINDOW_SEC = 1.0            # document default aggregation window

auto.require_environment()
print("[ENVIRONMENT OK] Gazebo, PX4, MAVROS, OMNeT++ and Simu5G are available")
print("[OUTPUT]", OUT.relative_to(HERE))

## 4. Fly a benign mission

### Listing 4 - minimal benign scenario manifest

The manifest carries scenario metadata and label windows. Labels may come from
here; every telemetry value must come from measurement.

In [ ]:
def manifest_for(scenario_id, family, attack=None, windows=None):
    """Build one scenario manifest in the document's Listing 4 shape."""
    document = {
        "organization_id": "org_01",
        "fleet_id": "fleet_09",
        "drone_id": "UAS-0063",
        "mission_id": scenario_id,
        "domain": "public_safety",
        "mission_type": "surveillance",
        "scenario_id": scenario_id,
        "scenario_family": family,
        "simulation": {
            "drone_count": 1,
            "mission_speed_mps": SPEED_MPS,
            "duration_sec": DURATION_SEC,
            "aggregation_window_sec": WINDOW_SEC,
            "cells": 1,
            "background_traffic": "low",
            "network_slice_profile": "single_mission_slice",
        },
        "attack": attack or {"enabled": False, "attack_type": "none"},
        "label_windows": windows or [{
            "start_sec": 0, "end_sec": DURATION_SEC,
            "attack_type": "none", "attack_stage": "none", "severity": "none",
            "incident_label": "normal",
            "recommended_response": "continue_monitoring",
        }],
        "allowed_destinations": ["10.0.0.2"],
    }
    return document


BENIGN = "B_DOC1_BENIGN_001"
benign_manifest = manifest_for(BENIGN, "benign")
manifest_dir = OUT / "manifests"
manifest_dir.mkdir(exist_ok=True)
(manifest_dir / f"{BENIGN}.yaml").write_text(
    yaml.safe_dump(benign_manifest, sort_keys=False), encoding="utf-8")
print(yaml.safe_dump(benign_manifest, sort_keys=False))

### Listing 5 - record the benign flight

Starts Gazebo, PX4 and MAVROS headless, records the three required MAVROS
topics to a ROS 2 bag, flies spawn to goal, lands and disarms.

Takes about five minutes. Do not run another Gazebo, PX4 or MAVROS meanwhile.

In [ ]:
run_dir = OUT / BENIGN
run_dir.mkdir(parents=True, exist_ok=True)

# start_simulation_stack launches headless Gazebo, PX4, MAVROS and the ROS 2
# recorder. It is the same helper the unattended duration tests use.
processes = []
try:
    processes, bag_dir = auto.start_simulation_stack(run_dir)
    print("[FLIGHT] spawn -> waypoints -> goal -> land")
    flight = auto.fly_for_duration(run_dir, DURATION_SEC, SPEED_MPS, ALTITUDE_M)
    print(f"[LANDED] active {flight['active_duration_sec']:.1f} s, "
          f"goal error {flight['final_goal_horizontal_error_m']:.3f} m, "
          f"landed and disarmed: {flight['landed_and_disarmed']}")
finally:
    for process in reversed(processes):
        process.stop()
print("[STOPPED] Gazebo, PX4, MAVROS and the recorder are shut down")

In [ ]:
# Checkpoint: a ROS 2 bag, per-topic rate and size measurements, and a pose
# trace with timestamped x/y/z. The document requires all three.
ros = auto.extract_ros_evidence(run_dir, run_dir / "rosbag")
print("[ROS TOPICS]")
for topic, stats in ros.items():
    print(f"  {topic:38s} {stats['message_count']:6d} msgs  "
          f"{stats['frequency_hz']:7.2f} Hz")

trace = OUT / "trace" / f"{BENIGN}_pose.txt"
movements = run_dir / "network/input/uav.movements"
movements.parent.mkdir(parents=True, exist_ok=True)
export = subprocess.run(
    [sys.executable, str(BRIDGE / "export_pose_trace.py"),
     str(run_dir / "ros_pose_trace.csv"), str(trace),
     "--movements", str(movements), "--duration", str(DURATION_SEC)],
    capture_output=True, text=True)
assert export.returncode == 0, export.stdout + export.stderr
print(export.stdout.strip())
print("[CHECKPOINT] bag, topic measurements and pose trace are present")

## 5. Configure the 5G network scenario

### Listing 6 - run Simu5G and export network metrics

One drone, one gNB cell, one 5G core, one edge node and one mission backend.
The measured pose drives the radio model, so latency, loss and signal quality
follow the flight rather than being invented.

In [ ]:
def build_scenario(scenario_id, manifest, controlled=None):
    """Write one Simu5G scenario directory and return its input path."""
    scenario_dir = OUT / scenario_id / "network"
    input_dir = scenario_dir / "input"
    for child in ("input", "raw", "data"):
        (scenario_dir / child).mkdir(parents=True, exist_ok=True)

    for name in ("EiraXExperiment.ned", "demo.xml", "flow_contract.json"):
        shutil.copy2(auto.NETWORK_TEMPLATE / name, input_dir / name)
    shutil.copy2(movements, input_dir / "uav.movements")

    ini = (auto.NETWORK_TEMPLATE / "omnetpp.ini").read_text(encoding="utf-8")
    ini = ini.replace("sim-time-limit = 1200s", f"sim-time-limit = {DURATION_SEC}s")
    ini = ini.replace("finishTime = 1199.9s", f"finishTime = {DURATION_SEC - 0.1:.1f}s")
    (input_dir / "omnetpp.ini").write_text(ini, encoding="utf-8")

    (input_dir / "scenario_manifest.yaml").write_text(
        yaml.safe_dump(manifest, sort_keys=False), encoding="utf-8")
    if controlled:
        (input_dir / "controlled_flow_contract.json").write_text(
            json.dumps(controlled, indent=1), encoding="utf-8")
    return input_dir


def run_scenario(scenario_id, manifest, controlled=None):
    """Listings 6 and 8: run Simu5G, export metrics, assemble, validate."""
    input_dir = build_scenario(scenario_id, manifest, controlled)
    scenario_dir = input_dir.parent
    raw_dir, data_dir = scenario_dir / "raw", scenario_dir / "data"

    ned = ":".join([str(auto.SIMU5G / "simulations"), str(auto.SIMU5G / "emulation"),
                    str(auto.SIMU5G / "src"), str(auto.SIMU5G.parent / "inet4.5/src"),
                    str(input_dir)])
    script = (f"set -e; source {auto.OMNETPP}/setenv -q; source {auto.INET}/setenv -q; "
              f"cd {auto.SIMU5G}; source ./setenv -f; cd {input_dir}; "
              f"simu5g -n {ned} -u Cmdenv -r 0 --result-dir={raw_dir}; "
              f"opp_scavetool x {raw_dir}/baseline.sca {raw_dir}/baseline.vec "
              f"-o {raw_dir}/network_metrics.csv")
    log = scenario_dir.parent / "simu5g.log"
    with log.open("w", encoding="utf-8") as handle:
        result = subprocess.run(["bash", "-lc", script], stdout=handle,
                                stderr=subprocess.STDOUT)
    assert result.returncode == 0, f"Simu5G failed; see {log}"

    output = data_dir / f"{scenario_id}_cosim.csv"
    assemble = subprocess.run(
        [sys.executable, str(BRIDGE / "assemble.py"), "--window", str(WINDOW_SEC),
         "--ros-log", str(run_dir), "--net-csv", str(raw_dir / "network_metrics.csv"),
         "--manifest", str(input_dir / "scenario_manifest.yaml"),
         "--flow-contract", str(input_dir / "flow_contract.json"),
         "--bridge-log", str(scenario_dir / "bridge_log.json"),
         "--schema", str(SCHEMA), "--out", str(output)],
        capture_output=True, text=True)
    assert assemble.returncode == 0, assemble.stdout + assemble.stderr

    check = subprocess.run([sys.executable, str(VALIDATOR), str(output), str(SCHEMA)],
                           capture_output=True, text=True)
    verdict = (check.stdout + check.stderr).strip().splitlines()[-1]
    assert check.returncode == 0, verdict
    return output, verdict


baseline_csv, verdict = run_scenario(BENIGN, benign_manifest)
print("[SIMU5G + BRIDGE]", baseline_csv.relative_to(HERE))
print("[VALIDATOR]", verdict)

## 6. The Python data bridge

`sim/bridge/assemble.py` produced the file above. It emits one row per flow per
aggregation window and derives every feature from measured Simu5G vectors
before any label is attached, which is what keeps the dataset usable for a
meaningful comparison.

In [ ]:
frame = pd.read_csv(baseline_csv)
columns = json.loads(SCHEMA.read_text(encoding="utf-8"))["columns"]
assert list(frame.columns) == columns, "column names or order changed"
print(f"[SHAPE] {len(frame)} rows x {len(frame.columns)} columns")
print(f"[WINDOW] {WINDOW_SEC} s, {frame.timestamp.nunique()} windows, "
      f"{frame.groupby(['source_port', 'destination_port']).ngroups} flows")
print("[LABELS]", dict(frame.incident_label.value_counts()))
frame.head(3)

## 7. Add suspicious and malicious scenarios

Table 1 of the document. One class at a time; validate after every run. The
suspicious run injects no attack, only ambiguity.

In [ ]:
ATTACK_START, ATTACK_END = 120, 180
ROGUE_IP = "10.0.0.99"


def attack_windows(attack_type, severity, response):
    """Label only the attack interval, never the whole run."""
    return [
        {"start_sec": 0, "end_sec": ATTACK_START, "attack_type": "none",
         "attack_stage": "none", "severity": "none", "incident_label": "normal",
         "recommended_response": "continue_monitoring"},
        {"start_sec": ATTACK_START, "end_sec": ATTACK_END, "attack_type": attack_type,
         "attack_stage": "execution", "severity": severity,
         "incident_label": "suspicious" if attack_type == "unknown_anomaly" else "malicious",
         "recommended_response": response},
        {"start_sec": ATTACK_END, "end_sec": DURATION_SEC, "attack_type": "none",
         "attack_stage": "none", "severity": "none", "incident_label": "normal",
         "recommended_response": "continue_monitoring"},
    ]


def rogue(**overrides):
    """A controlled flow injected on top of the benign profile."""
    flow = {"rate": 2.0, "size": 256, "start": ATTACK_START, "end": ATTACK_END,
            "source_ue": 1, "activity": "network", "source_component": "external_host",
            "source_ip": ROGUE_IP, "destination_component": "mission_backend",
            "destination_ip": "10.0.0.2", "protocol": "UDP", "receiver_app": 4}
    flow.update(overrides)
    return flow


SCENARIOS = [
    ("S_DOC1_ANOMALY_001", "suspicious", "unknown_anomaly", "low",
     "increase_monitoring",
     [rogue(name="ambiguous_retry", activity="authentication", protocol="HTTPS",
            source_component="drone", source_ip="10.0.0.1", source_port=7743,
            destination_port=8743, session_id="known-misconfigured-client",
            authentication_failed=True, known_destination=True)]),
    ("M_DOC1_DOS_001", "malicious", "denial_of_service", "high",
     "rate_limit_and_block_flow",
     [rogue(name="rogue_flood", rate=200.0, size=1200, source_port=7990,
            destination_port=8701, session_id="rogue-flood", known_destination=True)]),
    ("M_DOC1_SESSION_001", "malicious", "session_hijacking", "high",
     "revoke_session_force_reauth",
     [rogue(name="replayed_command_session", activity="command",
            protocol="MAVLink-like", source_port=7991, destination_port=8702,
            session_id="command-session", known_destination=True, session_reuse=True)]),
    ("M_DOC1_IDENTITY_001", "malicious", "identity_theft", "high",
     "suspend_identity_quarantine_device",
     [rogue(name="stolen_uav_identity", rate=1.0, size=512, activity="authentication",
            protocol="HTTPS", source_port=7992, destination_port=8992,
            session_id="stolen-identity", authentication_failed=True,
            claimed_drone_id="UAS-0063", network_slice_id="slice_4")]),
    ("M_DOC1_CONTROL_001", "malicious", "control_protocol_attack", "high",
     "block_command_channel_safe_mode",
     [rogue(name="unauthorized_command", rate=5.0, size=128, activity="command",
            protocol="MAVLink-like", destination_component="drone",
            destination_ip="10.0.0.1", source_port=7993, destination_port=8993,
            session_id="rogue-command")]),
    ("M_DOC1_LATERAL_001", "malicious", "lateral_movement", "high",
     "segment_slice_isolate_component",
     [rogue(name=f"closed_port_probe_{port}", rate=1.0, size=96, protocol="TCP",
            source_port=7900 + index, destination_port=port,
            session_id=f"probe-{port}", timeout_failed=True, receiver_app=4 + index)
      for index, port in enumerate((7100, 7101, 7102, 7103))]),
    ("M_DOC1_API_001", "malicious", "network_application_attack", "high",
     "rate_limit_api_alert_security_team",
     [rogue(name="malformed_api_flood", rate=200.0, size=800, activity="api_request",
            protocol="HTTP", source_port=7995, destination_port=8995,
            session_id="rogue-api", timeout_failed=True)]),
]
print(f"{len(SCENARIOS)} scenarios: one suspicious plus the six attack classes")

In [ ]:
# Run one class at a time and validate after each, as the document requires.
outputs = {BENIGN: baseline_csv}
for scenario_id, family, attack_type, severity, response, flows in SCENARIOS:
    manifest = manifest_for(
        scenario_id, family,
        attack={"enabled": True, "attack_type": attack_type,
                "injection_point": flows[0]["name"], "start_sec": ATTACK_START,
                "end_sec": ATTACK_END, "intensity": severity},
        windows=attack_windows(attack_type, severity, response))
    (manifest_dir / f"{scenario_id}.yaml").write_text(
        yaml.safe_dump(manifest, sort_keys=False), encoding="utf-8")
    path, verdict = run_scenario(scenario_id, manifest, flows)
    outputs[scenario_id] = path
    print(f"  {scenario_id:22s} {verdict}")
print(f"\n[SCENARIOS COMPLETE] {len(outputs)} validated runs")

## 8. Validate and deliver the dataset

### Listing 9 - concatenate the runs and validate the final CSV

In [ ]:
final_csv = OUT / "data/drone_network_telemetry_cosim.csv"
final_csv.parent.mkdir(parents=True, exist_ok=True)
concat = subprocess.run(
    [sys.executable, str(BRIDGE / "concat.py"), *map(str, outputs.values()),
     "--schema", str(SCHEMA), "--out", str(final_csv)],
    capture_output=True, text=True)
assert concat.returncode == 0, concat.stdout + concat.stderr
print(concat.stdout.strip())

check = subprocess.run([sys.executable, str(VALIDATOR), str(final_csv), str(SCHEMA)],
                       capture_output=True, text=True)
print("[VALIDATOR]", (check.stdout + check.stderr).strip().splitlines()[-1])
assert check.returncode == 0

In [ ]:
# The document asks for accuracy, macro-F1, confusion matrices and feature
# importances. evaluate_models.py produces them without touching the schema.
report_dir = OUT / "reports"
metrics = subprocess.run(
    [sys.executable, str(HERE / "evaluate_models.py"), "--dataset", str(final_csv),
     "--reports", str(report_dir), "--name", "document1_model_comparison",
     "--split", ""],
    capture_output=True, text=True, cwd=HERE)
print(metrics.stdout.strip() or metrics.stderr.strip()[-800:])

## Final deliverables checklist

The eight items the setup document closes with.

In [ ]:
frame = pd.read_csv(final_csv)
bag_ok = (run_dir / "rosbag/metadata.yaml").is_file()
deliverables = {
    "Gazebo/PX4 flies a short waypoint mission": (run_dir / "px4.log").is_file(),
    "ROS 2 bags recorded for benign, suspicious and malicious runs":
        bag_ok and len(outputs) == 8,
    "Pose trace exported and used for Simu5G mobility":
        trace.is_file() and movements.is_file(),
    "Simu5G produces network metric CSV files":
        all((OUT / s / "network/raw/network_metrics.csv").is_file() for s in outputs),
    "Bridge creates one row per flow per aggregation window":
        len(frame) == sum(len(pd.read_csv(p)) for p in outputs.values()),
    "Behavioural flags derived from telemetry, not labels":
        bool(frame.loc[frame.traffic_burst_flag == 1, "incident_label"].nunique() > 1)
        or int(frame.traffic_burst_flag.sum()) > 0,
    "Final CSV passes validate_dataset.py": check.returncode == 0,
    "Baseline metrics reported":
        (report_dir / "document1_model_comparison.md").is_file(),
}
for name, passed in deliverables.items():
    print("[PASSED]" if passed else "[FAILED]", name)
assert all(deliverables.values()), "unmet deliverables"
print(f"\n[DOCUMENT 1 COMPLETE] {len(frame)} rows, {len(frame.columns)} columns, "
      f"{len(outputs)} scenarios")